<a href="http://landlab.github.io"><img style="float: left" src="https://raw.githubusercontent.com/landlab/tutorials/release/landlab_header.png"></a>

# <span style="color:red">Team ESPIn! Let's make a two-component rainfall-infiltration-runoff-overland flow model!</span>

This tutorial illustrates the `SoilInfiltrationGreenAmpt` and `KinwaveImplicitOverlandFlow` components. You should already be familiar with Landlab grids, fields, and components before running this notebook.

This notebook illustrates how to use each component separately. You will then couple the components on your own!

*This notebook was modified from notebook originally written by Greg Tucker, September 2021.
The original notebook is part of the Landlab tutorial library, and can be found at:
https://landlab.csdms.io/tutorials/overland_flow/soil_infiltration_green_ampt/infilt_green_ampt_with_overland_flow.html
The notebook was modified for ESPIn, 2026.*

### The Green-Ampt infiltration component

The Green-Ampt method was introduced by Green and Ampt (1911) as a means of approximating the rate of water infiltration into soil from a layer of surface water. The method represents infiltration in terms of a wetting front that descends into the soil as infiltration progresses. A description of the method can be found in many hydrology textbooks, and in various online resources. The following is a brief summary, using the notation of Julien et al. (1995). The dimensions of each variable are indicated in square brackets, using the common convention that [L] means length, [M] is mass, and [T] is time.

The Green-Ampt method approximates the rate of water infiltration into the soil, $f$ (dimensions of [L/T], representing water volume per unit surface area). Infiltration is driven by two effects:  gravitational force, and downward suction (the "paper towel effect") due to a gradient in moisture at the wetting front. The method treats the infiltration rate as a function of the following parameters:

- $K$ - saturated hydraulic conductivity [L/T]
- $H_f$ - capillary pressure head at the wetting front [L]
- $\phi$ - total soil porosity [-]
- $\theta_r$ - residual saturation [-]
- $\theta_e$ - effective porosity $= \phi - \theta_r$ [-]
- $\theta_i$ - initial soil moisture content [-]
- $M_d$ - moisture deficit $=\theta_e - \theta_i$ [-]
- $F$ - total infiltrated water depth [L]

The equation for infiltration rate is:

$$f = K \left( 1 + \frac{H_fM_d}{F} \right)$$

The first term in parentheses represents gravity and the second represents pore suction. If there were no pore suction effect, water would simply infiltrate downward at a rate equal to the  hydraulic conductivity, $K$. The suction effect increases this, but it becomes weaker as the cumulative infiltration depth $F$ grows. Effectively, the second term approximates the pore-pressure gradient, which declines as the wetting front descends.

The version used in this component adds a term for the weight of the surface water with depth $H$:

$$f = K \left( 1 + \frac{H_fM_d}{F} + \frac{H}{F} \right)$$

The component uses a simple forward-difference numerical scheme, with time step duration $\Delta t$, in which the infiltration depth during one step is the lesser of the rate calculated above times $\Delta t$, or the available surface water, $H$:

$$\Delta F = \min( f\Delta t, H)$$

Note that the cumulative infitration $F$ must be greater than zero in order to avoid division by zero; therefore, one should initialize the `soil_water_infiltration__depth` to a small positive value.

### Import statements

Code block 0.1 should be run first and does not need to be rerun 
during continual use of this notebook.

In [ ]:
# Code block 0.1

from landlab.components import KinwaveImplicitOverlandFlow, SoilInfiltrationGreenAmpt
from landlab import RasterModelGrid
from landlab.io import esri_ascii

### Read in topography from a sample DEM

We will use a lidar digital elevation model (DEM) from the West Bijou Creek escarpment on the Colorado High Plains, coarsened to 5 m grid resolution, for all examples in this notebook.

We read in the DEM and create `base_grid` here. 

Code block 0.2 should be run after code block 0.1. It does not need to be rerun during continual use of this notebook.

In [ ]:
# Code block 0.2

# Create Landlab grid with DEM elevations.
# Display the terrain.
# In later examples we will create new grids
# with the same elevation values.

with open("../data/bijou_gully_subset_5m_edit_dx_filled.asc") as fp:
    base_grid = esri_ascii.load(fp, name="topographic__elevation", at="node")

# information we will need later to create copy grids
z_vals = base_grid.at_node["topographic__elevation"]
num_rows = base_grid.number_of_node_rows
num_cols = base_grid.number_of_node_columns
node_spacing = 5.0

# make a map
base_grid.imshow(
    base_grid.at_node["topographic__elevation"], colorbar_label="Elevation (m)"
)

## <span style="color:green">Example with infiltration only</span>

### Put an absurd amount of water on the landscape and see how much infiltrates

Start by making a new grid (see comment below), initializing variables, and instantiating the infiltation component with a set hydraulic conductivity.

In [ ]:
# We make a new grid with the same z values here.
# The reason for this is that if you want to do the same model
# experiment again with different parameters or initial values,
# the easiest way to do that is by making a new grid.

grid = RasterModelGrid((num_rows, num_cols), node_spacing)
grid.at_node["topographic__elevation"] = z_vals

# Create and initialize required input fields for infiltration
# component: depth of surface water, and depth (water volume per
# area) of initial infiltrated water. Note that units are in meters

depth = grid.add_zeros("surface_water__depth", at="node")
depth[:] = 10  # large initial surface water depth (10 m)

infilt = grid.add_zeros("soil_water_infiltration__depth", at="node")
infilt[:] = 1.0e-4  # small initial amount (0.1 mm)

# Instantiate an infiltration component, use initial hydraulic conductivity value of 5e-6 m/sec
ga = SoilInfiltrationGreenAmpt(grid, hydraulic_conductivity=5.0e-6)

### Plot grids of initial infiltration and surface water depths

In [ ]:
grid.imshow(
    1000.0 * infilt, colorbar_label="initial infiltration depth (mm)", cmap="GnBu"
)

In [ ]:
grid.imshow(
    "surface_water__depth",
    colorbar_label="initial surface water depth (m)",
    cmap="GnBu",
)

### Set time step and model duration

In [ ]:
dt = 10.0  # time step, sec
model_duration = 600.0  # model duration, sec

nsteps = int(model_duration / dt)

### Run the infiltration model

In [ ]:
# Model loop
for i in range(nsteps):
    ga.run_one_step(dt)

### Plot the cumulative infiltration and new surface water depth

In [ ]:
grid.imshow(
    1000.0 * infilt,
    colorbar_label="infiltration depth after model run (mm)",
    cmap="GnBu",
)

In [ ]:
grid.imshow(
    "surface_water__depth",
    colorbar_label="surface water depth after model run (m)",
    cmap="GnBu",
)

### Reflect

Admittedly this is a boring and ridiculous example. Nevertheless, it illustrates what the model does. Does it seem like water is being conserved? (hopefully!)

### *<span style="color:red">Challenge yourself</span>*

Rerun the model with a hydraulic conductivity that is an order of magnitude different and see what happens? Does it make sense?

### Do you want to explore the component more?

The `SoilInfiltrationGreenAmpt` component provides a variety parameters that can be set by the user. A list and description of these can be found in the component's `__init__` docstring, which is printed below:

In [ ]:
print(SoilInfiltrationGreenAmpt.__init__.__doc__)

## The kinematic wave component

The kinematic wave equations are a simplified form of the 2D shallow-water equations in which energy slope is assumed to equal bed slope. Conservation of water mass is expressed in terms of the time derivative of the local water depth, $H$, and the spatial derivative (divergence) of the unit discharge vector $\mathbf{q} = UH$ (where $U$ is the 2D depth-averaged velocity vector):

$$\frac{\partial H}{\partial t} = R - \nabla\cdot \mathbf{q}$$

where $R$ is the local runoff rate [L/T] and $\mathbf{q}$ has dimensions of volume flow per time per width [L$^2$/T]. The discharge depends on the local depth, bed-surface gradient $\mathbf{S}=-\nabla\eta$ (this is the kinematic wave approximation; $\eta$ is land surface height), and a roughness factor $C_r$:

$$\mathbf{q} = \frac{1}{C_r} \mathbf{S} H^\alpha |S|^{-1/2}$$

Readers may recognize this as a form of the Manning, Chezy, or Darcy-Weisbach equation. If $\alpha = 5/3$ then we have the Manning equation, and $C_r = n$ is "Manning's n". If $\alpha = 3/2$ then we have the Chezy/Darcy-Weisbach equation, and $C_r = 1/C = (f/8g)^{1/2}$ represents the Chezy roughness factor $C$ and the equivalent Darcy-Weisbach factor $f$.

### Numerical solution

The solution method used by this component is locally implicit, and works as follows. At each time step, we iterate from upstream to downstream over the topography. Because we are working downstream, we can assume that we know the total water inflow to a given cell. We solve the following mass conservation equation at each cell:

$$\frac{H^{t+1} - H^t}{\Delta t }= \frac{Q_{in}}{A} - \frac{Q_{out}}{A} + R$$

where $H$ is water depth at a given grid node, $t$ indicates time step number, $\Delta t$ is time step duration, $Q_{in}$ is total inflow discharge, $Q_{out}$ is total outflow discharge, $A$ is cell area, and $R$ is local runoff rate (precipitation minus infiltration; could be negative if runon infiltration is occurring).

The specific outflow discharge leaving a cell along one of its faces is:

$$q = (1/C_r) H^\alpha S^{1/2}$$

where $S$ is the downhill-positive gradient of the link that crosses this particular face. Outflow discharge is zero for links that are flat or "uphill" from the given node. Total discharge out of a cell is then the sum of (specific discharge x face width) over all outflow faces:

$$Q_{out} = \sum_{i=1}^N (1/C_r) H^\alpha S_i^{1/2} W_i$$

where $N$ is the number of outflow faces (i.e., faces where the ground slopes downhill away from the cell's node), and $W_i$ is the width of face $i$.

We use the depth at the cell's node, so this simplifies to:

$$Q_{out} = (1/C_r) H'^\alpha \sum_{i=1}^N S_i^{1/2} W_i$$

Notice that we know everything here except $H'$. The reason we know $Q_{out}$ is that it equals $Q_{in}$ (which is either zero or we calculated it previously) plus $RA$.

We define $H$ in the above as a weighted sum of the "old" (time step $t$) and "new" (time step $t+1$) depth values:

$$H' = w H^{t+1} + (1-w) H^t$$

If $w=1$, the method is fully implicit. If $w=0$, it is a simple forward explicit method.

When we combine these equations, we have an equation that includes the unknown $H^{t+1}$ and a bunch of terms that are known. If $w\ne 0$, it is a nonlinear equation in $H^{t+1}$, and must be solved iteratively. We do this using a root-finding method in the scipy.optimize library.

## <span style="color:green">Example with overland flow only</span>

### *<span style="color:red">First, challenge yourself!</span>*

In the code block below, write the code to display all the parameter values available for the user to set in the `KinwaveImplicitOverlandFlow` component.

In [ ]:
# YOU write the code!
# Find out what parameter values can be set
# in KinwaveImplicitOverlandFlow component

### Create a new grid to operate on and instantiate the component.

Because we use the same name for the grid, i.e. `grid`, we are overwriting
everything above (if you ran the infiltation only example).
If you want to save that grid, you would need to create a grid 
with a different name and update the rest of this example.

In [ ]:
grid = RasterModelGrid((num_rows, num_cols), node_spacing)
grid.at_node["topographic__elevation"] = z_vals

depth = grid.add_zeros("surface_water__depth", at="node")

# Instantiate an overland flow component
kw = KinwaveImplicitOverlandFlow(
    grid, runoff_rate=90.0, roughness=0.1, depth_exp=5.0 / 3.0
)

### Set time step and storm duration

In [ ]:
dt = 10.0  # time step, sec
storm_duration = 300.0  # storm duration, sec

nsteps = int(storm_duration / dt)

### Run the overland flow model

In [ ]:
# Run the model for the duration set in storm_duration

for i in range(nsteps):
    kw.run_one_step(dt)

### Plot a grid of surface water depth

Does it make any sense? you can refer back to the very first plot of topography in this notebook.

In [ ]:
grid.imshow(
    grid.at_node["surface_water__depth"], cmap="Blues", colorbar_label="Water depth (m)"
)

### *<span style="color:red">Challenge yourself</span>*

Can you rerun the overland flow model starting with 10 cm of initial surface water?
Can you rerun the overland flow model with double the amount of rainfall?

# *<span style="color:red">The big challenge! A coupled model.</span>*

### <span style="color:green">Using the same DEM as above, couple Green Ampt infiltration with the kinimatic wave component.</span>

Details:
- Initialize the surface water depth with 10 cm everywhere
- Initialize the infiltrated water depth as 1 cm everywhere
- Use an hydraulic conductivity value of 1e-6 m/s
- Run the model with a precipitation rate of 100 mm/hr
- Run the model for 10 minutes

Plots:
- Grids of the initial surface water and soil water depths
- Grids of the post-run surface water and soil water depths

*You got this!* But if you get stuck, a similar model with slightly different parameter values can be found in the Landlab documentation [here](https://landlab.csdms.io/tutorials/overland_flow/soil_infiltration_green_ampt/infilt_green_ampt_with_overland_flow.html) 

In [ ]:
# Your coupled model code goes here!

## References

Green, W. H., & Ampt, G. A. (1911). Studies on Soil Phyics. The Journal of Agricultural Science, 4(1), 1-24.

Julien, P. Y., Saghaﬁan, B., and Ogden, F. L. (1995) Raster-based hydrologic modeling of spatially-varied surface runoff, J. Am. Water Resour. As., 31, 523–536, doi:10.1111/j.17521688.1995.tb04039.x.

Rengers, F. K., McGuire, L. A., Kean, J. W., Staley, D. M., and Hobley, D. (2016) Model simulations of flood and debris flow timing in steep catchments after wildfire, Water Resour. Res., 52, 6041–6061, doi:10.1002/2015WR018176.